# Description
Run `database_prep.ipynb` before running this script.

# Imports

In [12]:
import pandas as pd
import os

In [13]:
root_dir = "/Users/emmanuelle.coutu-nadeau/Library/Mobile Documents/com~apple~CloudDocs/UdeM/MSc Psycho/LABO NED - Personal Drive/Code/GENiAL/"
og_data = os.path.join(root_dir, 'Data/Final/GENIAL-DB-preprocessed-V2.csv') # original data
og_data = pd.read_csv(og_data)

# Data Manipulations

Make sure EEG features are numerical

In [14]:
# Make sure EEG_ columns are numeric
# Get all EEG_ columns except known non-numeric ones
non_numeric_cols = ['EEG_attempted', 'EEG_site', 'EEG_date', 'EEG_age', 'EEG_Age', 'EEG_Sex']
eeg_cols = [col for col in og_data.columns if col.startswith('EEG_') and col not in non_numeric_cols]

# Convert EEG columns to numeric, coercing errors to NaN
for col in eeg_cols:
    og_data[col] = pd.to_numeric(og_data[col], errors='coerce')


Remove over 80% EEG features issing rows

In [17]:
# Calculate the percentage of missing values for each row
missing_percentage = og_data[eeg_cols].isnull().mean(axis=1)

# Keep only rows where less than 80% of EEG features are missing
no_missing_data = og_data[missing_percentage < 0.8].reset_index(drop=True)

print(f"Rows remaining after dropping those with >80% missing EEG data: {len(no_missing_data)}")


Rows remaining after dropping those with >80% missing EEG data: 80


In [19]:
diagnostic_groups_data = no_missing_data.copy()

# Add diagnostic group column
# 0: Control (diag_control = 1)
# 1: Neurodev only (diag_neurodev = 1 and diag_genetic_carrier = 0)
# 2: Genetic carrier (diag_genetic_carrier = 1)
diagnostic_groups_data['diagnostic_group'] = 0

# Set group 1: Neurodev only
diagnostic_groups_data.loc[(diagnostic_groups_data['diag_neurodev'] == 1) & (diagnostic_groups_data['diag_genetic_carrier'] == 0), 'diagnostic_group'] = 1

# Set group 2: Genetic carrier
diagnostic_groups_data.loc[diagnostic_groups_data['diag_genetic_carrier'] == 1, 'diagnostic_group'] = 2


# Stats

Summarize data

In [20]:
# Get summary statistics for EEG columns
eeg_summary = diagnostic_groups_data[eeg_cols].agg(['min', 'max', 'mean']).round(2)

# Display summary
print("\nEEG Features Summary:")
print(eeg_summary)



EEG Features Summary:
      EEG_Exponent-Frontal  EEG_Exponent-Central  EEG_Exponent-temporal-r  \
min                   0.54                  0.63                     0.52   
max                   1.98                  2.04                     1.99   
mean                  1.35                  1.40                     1.30   

      EEG_Exponent-temporal-l  EEG_Exponent-parietal-r  \
min                      0.31                     0.53   
max                      1.99                     1.98   
mean                     1.33                     1.33   

      EEG_Exponent-parietal-l  EEG_Exponent-Occipital  \
min                      0.27                    0.63   
max                      1.91                    2.04   
mean                     1.32                    1.46   

      EEG_Exponent-WholeBrain  EEG_Offset-Frontal  EEG_Offset-Central  ...  \
min                      0.66                0.15               -0.48  ...   
max                      1.94                2.26 

In [21]:
# Calculate z-scores for each EEG feature within each diagnostic group
z_scores = pd.DataFrame()

for group in diagnostic_groups_data['diagnostic_group'].unique():
    group_data = diagnostic_groups_data[diagnostic_groups_data['diagnostic_group'] == group]
    
    # Calculate z-scores for all EEG columns in this group
    group_z_scores = group_data[eeg_cols].apply(lambda x: (x - x.mean()) / x.std())
    
    # Add group identifier
    group_z_scores['diagnostic_group'] = group
    
    # Append to main z-scores dataframe
    z_scores = pd.concat([z_scores, group_z_scores])

# Reset index of final dataframe
z_scores = z_scores.reset_index(drop=True)

print("\nZ-scores calculated for each diagnostic group")
print(f"Shape of z-scores dataframe: {z_scores.shape}")




Z-scores calculated for each diagnostic group
Shape of z-scores dataframe: (80, 177)


In [31]:
# Check for extreme z-scores (|z| > 3.29)
extreme_mask = (z_scores[eeg_cols].abs() > 3.29)
num_extreme = extreme_mask.sum()

print("\nNumber of extreme z-scores (|z| > 3.29) for each EEG feature:")
print(num_extreme)

total_extreme_rows = extreme_mask.any(axis=1).sum()

if total_extreme_rows > 0:
    print(f"\nFound {total_extreme_rows} rows with extreme z-scores")
    print("\nBreakdown by diagnostic group:")
    for group in [0, 1, 2]:
        group_rows = extreme_mask[z_scores['diagnostic_group'] == group].any(axis=1).sum()
        group_name = {
            0: "Control",
            1: "Neurodev only", 
            2: "Genetic carrier"
        }[group]
        print(f"{group_name}: {group_rows} rows with extreme values")
else:
    print("\nNo extreme z-scores found.")



Number of extreme z-scores (|z| > 3.29) for each EEG feature:
EEG_Exponent-Frontal       0
EEG_Exponent-Central       0
EEG_Exponent-temporal-r    0
EEG_Exponent-temporal-l    0
EEG_Exponent-parietal-r    0
                          ..
EEG_DFA-Beta_temporal-l    0
EEG_DFA-Beta_parietal-r    0
EEG_DFA-Beta_parietal-l    0
EEG_DFA-Beta_Occipital     0
EEG_DFA-Beta_WholeBrain    0
Length: 176, dtype: int64

Found 18 rows with extreme z-scores

Breakdown by diagnostic group:
Control: 6 rows with extreme values
Neurodev only: 6 rows with extreme values
Genetic carrier: 6 rows with extreme values
